# UCS420 – Cognitive Computing | Assignment 10
## Diabetes Care Information Assistant

**Student:** Aakriti Gupta  
**Roll No.:** 1024170421 (Odd)  
**Batch:** 3Q23

This notebook implements the *Diabetes Care Information Assistant* assigned to students with odd roll numbers.

### Assignment requirements covered
- At least 40 labelled queries across 5 or more intents
- Hierarchical taxonomy with 4+ major categories and 2+ subcategories each
- 8+ domain concepts and 10+ relationships
- Lowercasing, tokenization, stopword removal and lemmatization
- TF-IDF and a Naive Bayes classifier
- Accuracy, precision, recall, F1-score and confusion matrix
- Domain-specific knowledge base and at least 5 reasoning rules
- Integrated assistant with an interactive multi-query interface
- Tests for defined intent, different wording, unsupported query, low confidence and emergency query

**Safety note:** This is an educational demonstration, not a diagnostic tool. It does not recommend medication doses or replace a healthcare professional. Emergency symptoms require urgent professional help.


## Cell 1 – Import libraries and download NLTK resources

In [ ]:
!pip -q install nltk scikit-learn pandas matplotlib seaborn ipywidgets

import re
import numpy as np
import pandas as pd
import nltk
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)

print("Libraries and NLTK resources are ready.")


## Cell 2 – Create a labelled corpus (100 queries, 10 intents)

In [ ]:
examples = {
    "diabetes_types": [
        "What are the main types of diabetes?",
        "Explain type one diabetes in simple words.",
        "What is type two diabetes?",
        "How is gestational diabetes different?",
        "Can diabetes occur during pregnancy?",
        "What is the difference between type one and type two?",
        "Tell me about diabetes categories.",
        "Is prediabetes the same as diabetes?",
        "What does type one diabetes mean?",
        "What does type two diabetes mean?"
    ],
    "diabetes_symptoms": [
        "What are common signs of diabetes?",
        "Why do people with diabetes feel thirsty?",
        "Can frequent urination be a symptom?",
        "What are early warning signs of diabetes?",
        "Can diabetes cause tiredness?",
        "Does blurred vision happen with diabetes?",
        "Why might someone feel hungry all the time?",
        "What symptoms can occur with high blood sugar?",
        "Can unexplained weight changes be a sign?",
        "How can I recognise possible diabetes symptoms?"
    ],
    "glucose_monitoring": [
        "How do I check my blood glucose at home?",
        "How does a glucose meter work?",
        "When should blood sugar be monitored?",
        "What is continuous glucose monitoring?",
        "How do I use a blood glucose meter?",
        "Why should I record glucose readings?",
        "What is a glucose sensor?",
        "How often should I check my glucose?",
        "What information does a glucose monitor show?",
        "How can I keep a blood sugar log?"
    ],
    "blood_sugar_levels": [
        "What does fasting blood glucose mean?",
        "What is an HbA1c test?",
        "What does an A1c result measure?",
        "How is blood glucose measured in a laboratory?",
        "What is a post meal glucose test?",
        "Why do doctors check HbA1c?",
        "What is the difference between fasting glucose and A1c?",
        "How do I understand a blood sugar report?",
        "What does a blood glucose reading represent?",
        "Who can explain my glucose test results?"
    ],
    "diabetes_diet": [
        "What is a balanced diet for diabetes?",
        "How can meal planning help manage diabetes?",
        "Why should carbohydrate portions be considered?",
        "What foods are high in dietary fibre?",
        "How does food affect blood glucose?",
        "What should a diabetes-friendly meal include?",
        "Why is portion size important?",
        "Can a dietitian help with diabetes meal planning?",
        "What are some healthy meal planning habits?",
        "How can I make balanced food choices?"
    ],
    "food_recommendations": [
        "Can people with diabetes eat whole fruits?",
        "Are vegetables suitable for a diabetes meal plan?",
        "Are whole grains a useful food choice?",
        "What are some fibre-rich foods?",
        "Can beans and lentils be included in meals?",
        "What are examples of balanced snacks?",
        "Should sugary drinks be limited?",
        "Are nuts part of a balanced eating pattern?",
        "Can I include brown rice in a meal plan?",
        "Which foods contain whole grains?"
    ],
    "physical_activity": [
        "Can walking support diabetes management?",
        "Why is exercise useful for people with diabetes?",
        "What kinds of physical activity can I try?",
        "Can cycling help with general fitness?",
        "How can I build an activity routine?",
        "Should I discuss exercise with my healthcare provider?",
        "Can activity affect blood glucose?",
        "What are examples of moderate exercise?",
        "How can I reduce long periods of sitting?",
        "What precautions should I consider before exercising?"
    ],
    "daily_care": [
        "What should daily diabetes self care include?",
        "Why is foot care important?",
        "Why should I attend regular checkups?",
        "How can I keep track of appointments?",
        "Why is taking prescribed medicine as directed important?",
        "How can I prepare for a diabetes appointment?",
        "What should I record in a health diary?",
        "How can I remember my care routine?",
        "Why should I check my feet regularly?",
        "Who can help me make a diabetes care plan?"
    ],
    "hypoglycemia": [
        "What is hypoglycemia?",
        "What does low blood sugar mean?",
        "What are signs of low glucose?",
        "Can shaking happen when blood sugar is low?",
        "Why can low blood sugar cause sweating?",
        "What should a conscious person do for low blood sugar according to their care plan?",
        "When is low blood sugar an emergency?",
        "Can confusion be a sign of low glucose?",
        "What is the meaning of a glucose reading below 70?",
        "Why should severe low blood sugar be treated urgently?"
    ],
    "hyperglycemia": [
        "What is hyperglycemia?",
        "What does high blood sugar mean?",
        "What are common signs of high glucose?",
        "Can high blood sugar cause increased thirst?",
        "Why might someone urinate frequently with high glucose?",
        "What can cause blood glucose to rise?",
        "When should high blood sugar be discussed with a clinician?",
        "What symptoms of high glucose need urgent attention?",
        "Can illness affect blood glucose levels?",
        "What is diabetic ketoacidosis?"
    ]
}

rows = [
    {"query": query, "intent": intent}
    for intent, queries in examples.items()
    for query in queries
]
df = pd.DataFrame(rows)

print("Total queries:", len(df))
print("Number of intents:", df["intent"].nunique())
display(df.groupby("intent").size().rename("query_count").to_frame())
display(df.head())


## Cell 3 – Inspect and validate the corpus

In [ ]:
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate queries:", df["query"].duplicated().sum())
print("\nIntent distribution:")
print(df["intent"].value_counts())

assert len(df) >= 40
assert df["intent"].nunique() >= 5
assert df["query"].isnull().sum() == 0


## Cell 4 – Create hierarchical taxonomy

In [ ]:
taxonomy = {
    "Diabetes Fundamentals": [
        "diabetes_types",
        "diabetes_symptoms"
    ],
    "Glucose Monitoring and Tests": [
        "glucose_monitoring",
        "blood_sugar_levels"
    ],
    "Nutrition": [
        "diabetes_diet",
        "food_recommendations"
    ],
    "Lifestyle and Self-Care": [
        "physical_activity",
        "daily_care"
    ],
    "Blood Glucose Safety": [
        "hypoglycemia",
        "hyperglycemia"
    ]
}

print("HIERARCHICAL TAXONOMY")
for major, subcategories in taxonomy.items():
    print("\n" + major)
    for subcategory in subcategories:
        print("  └──", subcategory)

print("\nMajor categories:", len(taxonomy))
print("Subcategories:", sum(len(v) for v in taxonomy.values()))


## Cell 5 – Define domain concepts

In [ ]:
domain_concepts = {
    "Diabetes": "A group of conditions affecting blood glucose regulation.",
    "Insulin": "A hormone that helps glucose move from the blood into cells.",
    "Blood Glucose": "The amount of glucose present in the blood.",
    "Pancreas": "An organ that produces insulin and other hormones.",
    "Glucose Meter": "A device used to measure blood glucose from a sample.",
    "Continuous Glucose Monitor": "A device that tracks glucose levels over time.",
    "HbA1c": "A blood test reflecting average glucose over roughly the previous three months.",
    "Healthy Eating Pattern": "A balanced pattern of food choices and portions.",
    "Physical Activity": "Body movement that supports general health and glucose management.",
    "Hypoglycemia": "Blood glucose that is lower than the target range; below 70 mg/dL is a common alert threshold.",
    "Hyperglycemia": "Blood glucose that is higher than the target range.",
    "Healthcare Professional": "A qualified clinician involved in health assessment and care."
}

for concept, description in domain_concepts.items():
    print(f"{concept}: {description}")

print("\nTotal domain concepts:", len(domain_concepts))


## Cell 6 – Define relationships between concepts

In [ ]:
relationships = [
    ("Pancreas", "produces", "Insulin"),
    ("Insulin", "helps regulate", "Blood Glucose"),
    ("Diabetes", "affects regulation of", "Blood Glucose"),
    ("Glucose Meter", "measures", "Blood Glucose"),
    ("Continuous Glucose Monitor", "tracks", "Blood Glucose"),
    ("HbA1c", "estimates recent average", "Blood Glucose"),
    ("Healthy Eating Pattern", "can support management of", "Diabetes"),
    ("Physical Activity", "can influence", "Blood Glucose"),
    ("Hypoglycemia", "refers to low", "Blood Glucose"),
    ("Hyperglycemia", "refers to high", "Blood Glucose"),
    ("Healthcare Professional", "interprets", "HbA1c"),
    ("Healthcare Professional", "supports care planning for", "Diabetes"),
    ("Diabetes", "may require monitoring with", "Glucose Meter"),
    ("Diabetes", "may be monitored using", "Continuous Glucose Monitor"),
    ("Physical Activity", "is part of", "Healthy Eating Pattern and lifestyle care")
]

for subject, relation, obj in relationships:
    print(f"{subject} --{relation}--> {obj}")

print("\nTotal relationships:", len(relationships))


## Cell 7 – Text preprocessing

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words and len(word) > 1]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

df["processed_query"] = df["query"].apply(preprocess_text)

display(df[["query", "processed_query", "intent"]].head(10))


## Cell 8 – Split data, fit TF-IDF on training data, and train Naive Bayes

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["processed_query"],
    df["intent"],
    test_size=0.20,
    random_state=42,
    stratify=df["intent"]
)

tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

model = MultinomialNB(alpha=0.3)
model.fit(X_train, y_train)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("TF-IDF features:", len(tfidf.get_feature_names_out()))
print("Naive Bayes classifier trained.")


## Cell 9 – Predict test intents

In [ ]:
y_pred = model.predict(X_test)

prediction_results = pd.DataFrame({
    "Query": X_test_text.values,
    "Actual Intent": y_test.values,
    "Predicted Intent": y_pred,
    "Correct": y_test.values == y_pred
})
display(prediction_results)


## Cell 10 – Evaluate the classifier

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print("MODEL EVALUATION")
print("----------------")
print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1-score : {f1:.3f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))


## Cell 11 – Display confusion matrix

In [ ]:
labels = sorted(df["intent"].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(13, 9))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=labels, yticklabels=labels
)
plt.xlabel("Predicted intent")
plt.ylabel("Actual intent")
plt.title("Confusion Matrix – Diabetes Care Assistant")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## Cell 12 – Build the domain-specific knowledge base

In [ ]:
knowledge_base = {
    "diabetes_types": (
        "The main types include type 1 diabetes, type 2 diabetes and gestational diabetes. "
        "They differ in causes and circumstances. A healthcare professional can explain "
        "which type applies to an individual."
    ),
    "diabetes_symptoms": (
        "Possible symptoms include increased thirst, frequent urination, increased hunger, "
        "fatigue and blurred vision. Some people have few or no noticeable symptoms. "
        "A healthcare professional can advise on appropriate testing."
    ),
    "glucose_monitoring": (
        "Blood glucose may be checked using a blood glucose meter or a continuous glucose "
        "monitor. Follow the device instructions and the monitoring schedule agreed with "
        "your healthcare professional."
    ),
    "blood_sugar_levels": (
        "Blood glucose tests include fasting glucose and HbA1c. Interpretation depends on "
        "the test, timing, medical history and individual circumstances. Ask a healthcare "
        "professional to interpret your results."
    ),
    "diabetes_diet": (
        "A balanced eating pattern often includes vegetables, fibre-rich foods, suitable "
        "protein sources and attention to carbohydrate portions. Individual meal planning "
        "is best discussed with a registered dietitian or healthcare professional."
    ),
    "food_recommendations": (
        "Vegetables, pulses, whole grains and whole fruits can fit into many balanced meal "
        "plans. Portion needs vary. A dietitian can help adapt food choices to personal "
        "health needs and preferences."
    ),
    "physical_activity": (
        "Activities such as walking, cycling and swimming may support general health and "
        "blood glucose management. Choose activity appropriate to your health and discuss "
        "new exercise plans with a healthcare professional when needed."
    ),
    "daily_care": (
        "Self-care may include following the care plan, taking prescribed medicines as "
        "directed, monitoring glucose as advised, foot care and attending regular checkups. "
        "Do not change prescribed treatment without professional advice."
    ),
    "hypoglycemia": (
        "Hypoglycemia means low blood glucose; below 70 mg/dL is a commonly used alert "
        "threshold. Symptoms can include shaking, sweating, dizziness or confusion. "
        "Follow the person's clinician-provided treatment plan. If they are unconscious, "
        "having a seizure or unable to swallow, do not give food or drink by mouth and "
        "seek emergency medical help immediately."
    ),
    "hyperglycemia": (
        "Hyperglycemia means high blood glucose. Possible symptoms include increased "
        "thirst, frequent urination and fatigue. Vomiting, abdominal pain, deep or difficult "
        "breathing, fruity-smelling breath or confusion can be warning signs of a serious "
        "emergency such as diabetic ketoacidosis; seek urgent medical care."
    )
}

print("Knowledge base categories:", len(knowledge_base))
for intent in knowledge_base:
    print("-", intent)


## Cell 13 – Define domain detection and reasoning rules

In [ ]:
diabetes_keywords = {
    "diabetes", "diabetic", "glucose", "blood sugar", "insulin",
    "pancreas", "hba1c", "hypoglycemia", "hyperglycemia",
    "glucose meter", "blood sugar", "foot care", "carbohydrate",
    "fasting glucose", "blood glucose", "thirsty", "thirst", "urination", "frequent urination", "tiredness", "fatigue", "hunger", "blurred vision"
}

emergency_keywords = [
    "unconscious", "not responding", "seizure", "unable to swallow",
    "difficulty breathing", "trouble breathing", "deep breathing",
    "confusion and vomiting", "fainted"
]

def is_diabetes_related(query):
    q = query.lower()
    return any(keyword in q for keyword in diabetes_keywords)

def apply_reasoning(query, predicted_intent=None, confidence=1.0):
    q = query.lower()

    # Rule 1: Emergency escalation takes priority.
    if any(keyword in q for keyword in emergency_keywords):
        return (
            "EMERGENCY ALERT: This may be a medical emergency. Contact local emergency "
            "services or seek immediate medical care. Do not delay to use this assistant."
        )

    # Rule 2: Reject queries outside the assistant's defined domain.
    if not is_diabetes_related(q):
        return (
            "I can answer general questions about diabetes care only. "
            "Please ask a diabetes-related question."
        )

    # Rule 3: Handle low-confidence predictions.
    if confidence < 0.25:
        return (
            "I am not confident enough to classify this question. Please rephrase it "
            "or consult a healthcare professional."
        )

    # Rule 4: Retrieve information for a recognised intent.
    if predicted_intent not in knowledge_base:
        return "I do not have suitable information for that topic."

    answer = knowledge_base[predicted_intent]

    # Rule 5: Add a medical safety disclaimer.
    answer += (
        "\n\nEducational information only: this assistant cannot diagnose or prescribe. "
        "Consult a qualified healthcare professional for personal advice."
    )
    return answer

print("Five reasoning rules are ready.")


## Cell 14 – Integrate preprocessing, classification, reasoning and retrieval

In [ ]:
def diabetes_assistant(user_query, show_details=True):
    if not user_query or not user_query.strip():
        return "Please enter a question."

    # Emergency escalation is checked before classification.
    if any(k in user_query.lower() for k in emergency_keywords):
        return apply_reasoning(user_query)

    if not is_diabetes_related(user_query):
        return apply_reasoning(user_query)

    processed = preprocess_text(user_query)
    query_vector = tfidf.transform([processed])
    probabilities = model.predict_proba(query_vector)[0]
    best_index = int(np.argmax(probabilities))
    predicted_intent = model.classes_[best_index]
    confidence = float(probabilities[best_index])

    if show_details:
        print("Predicted intent:", predicted_intent)
        print("Model confidence:", round(confidence, 3))

    return apply_reasoning(user_query, predicted_intent, confidence)

# Example
print(diabetes_assistant("What are common symptoms of diabetes?"))


## Cell 15 – Test all five required scenarios

In [ ]:
test_cases = [
    ("Defined intent", "What are common symptoms of diabetes?"),
    ("Different wording", "Why am I always thirsty and visiting the toilet?"),
    ("Unsupported healthcare query", "What are the treatment options for cancer?"),
    ("Low-confidence handling demonstration", "Diabetes insulin pump calibration settings",),
    ("Emergency-related query", "A person with diabetes is unconscious and not responding.")
]

for test_name, query in test_cases:
    print("\n" + "=" * 65)
    print("TEST:", test_name)
    print("USER:", query)

    if test_name == "Low-confidence handling demonstration":
        # Explicitly pass a low score to exercise the low-confidence rule.
        # This demonstrates the rule branch; it is not a model-generated score.
        response = apply_reasoning(
            query,
            predicted_intent="glucose_monitoring",
            confidence=0.10
        )
        print("Confidence supplied for rule demonstration: 0.10")
    else:
        response = diabetes_assistant(query)

    print("ASSISTANT:", response)


## Cell 16 – Interactive assistant (accepts multiple queries)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

query_box = widgets.Text(
    placeholder="Ask a diabetes-related question...",
    description="Question:",
    layout=widgets.Layout(width="85%")
)

ask_button = widgets.Button(
    description="Ask Assistant",
    button_style="success",
    icon="comment"
)

chat_output = widgets.Output()

def handle_question(button):
    query = query_box.value.strip()
    with chat_output:
        clear_output()
        if not query:
            print("Please enter a question.")
        else:
            print("DIABETES CARE INFORMATION ASSISTANT")
            print("-" * 40)
            print("You:", query)
            print("\nAssistant:", diabetes_assistant(query))
    query_box.value = ""

ask_button.on_click(handle_question)

display(widgets.VBox([
    widgets.HTML("<h2>Diabetes Care Information Assistant</h2>"),
    widgets.HTML("<p>Ask general questions about diabetes care. This is an educational tool, not medical advice.</p>"),
    query_box,
    ask_button,
    chat_output
]))


## Cell 17 – Final assignment requirement checklist

In [ ]:
requirements = {
    "40 or more labelled queries": len(df) >= 40,
    "5 or more intent categories": df["intent"].nunique() >= 5,
    "4 or more taxonomy categories": len(taxonomy) >= 4,
    "2 or more subcategories per major category": all(len(v) >= 2 for v in taxonomy.values()),
    "8 or more domain concepts": len(domain_concepts) >= 8,
    "10 or more domain relationships": len(relationships) >= 10,
    "Lowercase, tokenization, stopword removal, lemmatization": "processed_query" in df.columns,
    "TF-IDF feature extraction": X_train.shape[1] > 0,
    "Naive Bayes classifier": hasattr(model, "classes_"),
    "Accuracy, precision, recall and F1": all(v >= 0 for v in [accuracy, precision, recall, f1]),
    "Confusion matrix": cm.shape[0] > 0,
    "Knowledge base": len(knowledge_base) >= 5,
    "At least five reasoning rules": True,
    "Interactive multi-query interface": True
}

print("FINAL ASSIGNMENT CHECKLIST")
print("=" * 55)
for item, passed in requirements.items():
    print(f"{'PASS' if passed else 'CHECK'} - {item}")

print("=" * 55)
print(f"Completed checks: {sum(requirements.values())}/{len(requirements)}")
print("\nRemember to run every cell from top to bottom in a fresh Colab runtime.")
